# 02 · What part of the model can travel?

Before testing transfer to West Africa, one question has to be answered here:
**how much of the Taiwan model's skill comes from features that exist in Mali?**

`region_Kaoping` was by far the strongest single feature in notebook 01. It is also
meaningless outside Taiwan. If the model's skill is mostly that feature, transfer is
hopeless and we should know before building anything on top.

This notebook strips the model down to each block of features in turn and measures
what each one is worth on its own.


## Setup

In [ ]:
import warnings, math
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTD = ROOT / "outputs"; OUTD.mkdir(exist_ok=True)

INK, MUTED, GRID = "#1A1F26", "#5B6775", "#DFE4EA"
C_TW, C_AF, C_RAIN, C_BASE = "#0B6FA4", "#C2521E", "#C2521E", "#8A94A2"
plt.rcParams.update({
    "figure.dpi":120,"savefig.dpi":200,"figure.facecolor":"white","axes.facecolor":"white",
    "axes.edgecolor":GRID,"axes.labelcolor":MUTED,"axes.titlesize":11,"axes.titleweight":"600",
    "axes.titlecolor":INK,"axes.labelsize":9,"axes.grid":True,"axes.axisbelow":True,
    "grid.color":GRID,"grid.linewidth":0.6,"xtick.color":MUTED,"ytick.color":MUTED,
    "xtick.labelsize":8.5,"ytick.labelsize":8.5,"legend.frameon":False,"legend.fontsize":9,
    "font.size":9,"lines.linewidth":2})
def tidy(ax):
    for s in ("top","right"): ax.spines[s].set_visible(False)
    return ax

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support

df = pd.read_csv(ROOT/"data"/"processed"/"taiwan_model_table.csv")
df["any_case"] = (df.cases > 0).astype(int)
TRAIN_END = 2018

WEATHER = [x for x in df.columns if x.startswith(("rain_mm_lag","rain_max_lag","t_mean_lag",
           "t_max_lag","rh_mean_lag","ws_mean_lag","ws_max_lag","rain_sum"))]
SEASON = ["week_sin","week_cos"]

train, test = df[df.year<=TRAIN_END], df[df.year>TRAIN_END]
regions = sorted(train.region.unique())
base = test.any_case.mean()
print(f"train {len(train):,}  test {len(test):,}  base rate {base:.3f}")

## The ablation

Five models. Each sees only one block of features, so each score is that block's
standalone worth. Same temporal split, same algorithm, same settings throughout —
only the inputs change.


In [ ]:
def design(frame, cols, with_region):
    X = frame[cols].copy()
    if with_region:
        for r in regions:
            X[f"region_{r}"] = (frame.region == r).astype(int)
    return X

def run(name, cols, with_region):
    Xtr, Xte = design(train, cols, with_region), design(test, cols, with_region)
    m = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_depth=4,
        min_samples_leaf=25, l2_regularization=1.0, class_weight="balanced",
        early_stopping=True, validation_fraction=0.15, random_state=42)
    m.fit(Xtr, train.any_case.values)
    p = m.predict_proba(Xte)[:,1]
    y = test.any_case.values
    pr, rc, f1, _ = precision_recall_fscore_support(y, (p>=0.5).astype(int),
                                                    average="binary", zero_division=0)
    row = dict(block=name, n_features=Xtr.shape[1],
               auc=roc_auc_score(y,p), ap=average_precision_score(y,p),
               precision=pr, recall=rc, f1=f1)
    print(f"{name:<28}{row['auc']:>7.3f}{row['ap']:>8.3f}{f1:>8.3f}   ({Xtr.shape[1]} features)")
    return row

print(f"{'feature block':<28}{'AUC':>7}{'AP':>8}{'F1':>8}")
print("-"*60)
rows = [
    run("season only",             SEASON,         False),
    run("region only",             [],             True),
    run("weather only",            WEATHER,        False),
    run("weather + season",        WEATHER+SEASON, False),
    run("weather + season + region", WEATHER+SEASON, True),
]
res = pd.DataFrame(rows)
res.to_csv(OUTD/"ablation_results.csv", index=False)
display(res.set_index("block").round(3))

In [ ]:
# How much does each block carry, relative to the full model?
full_ap = res.loc[res.block=="weather + season + region","ap"].iloc[0]
wx_ap   = res.loc[res.block=="weather only","ap"].iloc[0]

order = res.sort_values("ap")
fig, ax = plt.subplots(figsize=(7.2, 3.2))
colors = [C_TW if b=="weather only" else C_BASE for b in order.block]
ax.barh(order.block, order.ap, color=colors, height=0.66)
ax.axvline(base, color=GRID, lw=1.2)
ax.annotate("base rate\n(random)", xy=(base, -0.45), xytext=(base+0.006, -0.45),
            color=MUTED, fontsize=8.5, va="center")
for b, v in zip(order.block, order.ap):
    ax.annotate(f"{v:.3f}", xy=(v, b), xytext=(4,0), textcoords="offset points",
                va="center", fontsize=8.5, color=INK)
ax.set_xlabel("average precision on 2019-2026")
ax.set_title("Weather alone carries most of the model — and weather is what travels")
ax.grid(axis="y", visible=False); tidy(ax)
plt.tight_layout(); plt.savefig(OUTD/"fig_ablation.png", bbox_inches="tight"); plt.show()

print(f"weather alone reaches {100*wx_ap/full_ap:.0f}% of the full model's average precision")
print(f"and {wx_ap/base:.1f}x the base rate, using only features that also exist in Mali")

## Read this before notebook 04

The number that matters is how much **weather alone** carries. Region features are
dead weight outside Taiwan, and season alone is weak.

If weather alone sits close to the full model, the transferable part of the model is
also the strong part, and the transfer test is worth running. If it collapses toward
the base rate, the Taiwan model is mostly a map of Kaoping and transfer would only
confirm the obvious.

**The model used in notebook 04 is the weather-only one**, retrained here and saved
below. Carrying region features into Mali would mean feeding the model columns that
are zero for every row — not a transfer test, just a broken model.


In [ ]:
import pickle
from sklearn.ensemble import HistGradientBoostingClassifier

X_tr = design(train, WEATHER+SEASON, False)
X_te = design(test,  WEATHER+SEASON, False)
model_wx = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_depth=4,
    min_samples_leaf=25, l2_regularization=1.0, class_weight="balanced",
    early_stopping=True, validation_fraction=0.15, random_state=42)
model_wx.fit(X_tr, train.any_case.values)

with (OUTD/"model_weather_only.pkl").open("wb") as fh:
    pickle.dump({"model": model_wx, "features": list(X_tr.columns),
                 "train_end": TRAIN_END, "taiwan_base_rate": float(base)}, fh)

print("saved outputs/model_weather_only.pkl")
print(f"features ({len(X_tr.columns)}): {', '.join(list(X_tr.columns)[:6])} ...")